# ライブ表示の試作

「ランタイム → すべてのセルを実行」で、各セルの進捗表示を確かめます。

In [ ]:
#@title 準備 { display-mode: "form" }

_TRITFLOW_SETUP_READY = False
globals().pop("_tf_cells", None)
import os, sys, json, time, hashlib, subprocess, importlib.util
from pathlib import Path
started = time.monotonic()
def step(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
SOURCE_URL = "https://github.com/PenguinEino/tritflow.git"
SOURCE_REF = os.environ.get("TRITFLOW_REF", "baebab61879379b59e0cd812663222f0c93d5d86")
def source_git(*args):
    return subprocess.check_output(["git", "-C", str(ROOT), *args], text=True).strip()
if IN_COLAB:
    ROOT = Path("/content/tritflow")
    import re
    if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_REF):
        raise ValueError("TRITFLOW_REF must be an immutable full 40-character Git commit")
    if not ROOT.exists():
        step("Downloading the TritFlow source from GitHub")
        subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", "--filter=blob:none",
                        SOURCE_URL, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", SOURCE_REF], check=True)
        subprocess.run(["git", "-C", str(ROOT), "switch", "--detach", SOURCE_REF], check=True)
    if not (ROOT/".git").is_dir():
        raise RuntimeError("/content/tritflow must be a git clone; use a fresh Colab runtime")
    if source_git("remote", "get-url", "origin").removesuffix(".git") != SOURCE_URL.removesuffix(".git"):
        raise RuntimeError("existing checkout has a different origin; use a fresh Colab runtime")
    SOURCE_COMMIT = source_git("rev-parse", "--verify", "HEAD^{commit}")
    requested_commit = source_git("rev-parse", "--verify", SOURCE_REF+"^{commit}")
    if SOURCE_COMMIT != requested_commit or source_git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("existing checkout differs from TRITFLOW_REF; use a fresh Colab runtime")
    print(f"Source: {SOURCE_URL} ref={SOURCE_REF} commit={SOURCE_COMMIT}", flush=True)
    if not (ROOT/"notebook/runtime_support.py").is_file():
        raise RuntimeError("TRITFLOW_REF predates this notebook's setup. Select its matching immutable release with TRITFLOW_REF in a fresh runtime.")
    sys.path.insert(0, str(ROOT/"notebook"))
    from runtime_support import SETUP_SCHEMA, setup_identity
    if SETUP_SCHEMA != "3":
        raise RuntimeError("The selected source commit predates the pinned runtime. Use the final submission commit with TRITFLOW_REF.")
    runtime = ROOT/"notebook/runtime.json"
    if (not runtime.is_file()
            or json.loads(runtime.read_text()).get("TRITFLOW_SETUP_SCHEMA") != SETUP_SCHEMA
            or json.loads(runtime.read_text()).get("TRITFLOW_SETUP_SHA256") != setup_identity(ROOT)):
        step("Installing KLayout, ngspice, Python packages and the TR-1um PDK (first run only, about 2 minutes)")
        subprocess.run(["bash", str(ROOT/"notebook/colab_setup.sh"), "--source", str(ROOT)], check=True)
    else:
        step("Tools are already installed in this runtime")
    os.environ.update(json.loads(runtime.read_text()))
    sys.path.insert(0, os.environ["TRITFLOW_PYTHON_TARGET"])
else:
    candidates = [Path(os.environ.get("TRITFLOW_ROOT", ".")), Path.cwd(), Path.cwd().parent]
    ROOT = next((p.resolve() for p in candidates if (p/"pyproject.toml").is_file()
                 and (p/"tritflow").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("run locally from the full checkout or set TRITFLOW_ROOT")
    SOURCE_COMMIT = source_git("rev-parse", "--verify", "HEAD^{commit}") if (ROOT/".git").exists() else "tracked-tree export (no .git)"
    print(f"Source: {ROOT} commit={SOURCE_COMMIT}")
runtime = ROOT/"notebook/runtime.json"
if not IN_COLAB and runtime.is_file():
    os.environ.update(json.loads(runtime.read_text()))
    sys.path.insert(0, os.environ["TRITFLOW_PYTHON_TARGET"])
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/"notebook"))
sys.path.append(str(ROOT/"figures"))   # the paper's figure scripts, reused by method cells
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
from runtime_support import check_runtime, print_tool_versions
step("Checking the Python packages, KLayout and ngspice")
check_runtime()
print_tool_versions()
_TRITFLOW_SETUP_READY = True
step(f"Setup complete: {time.monotonic()-started:.0f} s")

BUILD = Path(os.environ.get("TRITFLOW_NOTEBOOK_BUILD", ROOT/"build/notebook")).resolve()
from live import step, need, provides, install_error_view, LayoutView
from tritflow import flow
install_error_view()
provides("design", "1-1 HDLを読み込む")
provides("net", "1-2 合成する")
provides("lay", "1-3 配置配線する")


In [ ]:
# 1-1 HDLを読み込む
source = """
module first(input trit a, output trit y);
  assign y = a & ~a;   // & is min
endmodule
"""
with step("HDLを読み込む") as s:
    design = flow.loads(source, "first")
    s.done("1入力 · 1出力")

In [ ]:
# 1-2 合成する
need("design", why="合成する回路のHDLは、1-1で読み込みます。")
with step("合成") as s:
    net = design.synthesize()
    s.done(" · ".join(f"{cell}×{n}" for cell, n in net.report()["cells"].items()))

In [ ]:
# 1-3 配置配線する
need("net", why="このセルは、合成した回路 net を配置配線します。")
with step("配置配線", view=LayoutView()) as s:
    lay = net.place_and_route(output_dir=BUILD/"first", seed=0)
    s.done(f"{lay.stats['instance_count']}セル · {lay.stats['die_size_um'][0]:.0f} × {lay.stats['die_size_um'][1]:.0f} µm")

In [ ]:
# 1-4 DRCとLVS
need("lay", why="DRCとLVSは、1-3で作ったGDSを調べます。")
with step("DRC") as s:
    drc = lay.drc()
    s.done(f"違反 {drc['violations']}件")
with step("LVS") as s:
    lvs = lay.lvs()
    s.done("一致" if lvs["matched"] else "不一致")

In [ ]:
# 1-5 抽出後のSPICE
need("lay", why="SPICEは、1-3で作ったGDSから取り出した回路を動かします。")
vectors = [(-1,), (0,), (1,), (0,), (-1,)]
with step("抽出後SPICE") as s:
    post = lay.extract().simulate(vectors, load=100e-15, expected=lambda v: {"y": min(v[0], -v[0])},
                                  hold_s=1e-6, step_s=2e-9, edge_s=1e-9, temperature_c=27)
    s.done("全区間で合格" if post["measurements"]["passed"] else "不合格")

In [ ]:
# 1-6 振り返る
need("lay", why="配置配線の途中経過は、1-3の実行で記録されます。")
from tritflow.viz.layout import placement_animation, routing_progress
placement_animation(lay.data).show()
routing_progress(lay.data).show()

In [ ]:
# 順番を間違えた例：未定義の変数を使う
del lay
with step("DRC") as s:
    lay.drc()